# KV260 icin YOLOX-Nano - Havadan Arac Tespiti (Kaggle)

**Hedef:** 2 sinif - `land_vehicle`, `sea_vehicle`. Her tespitin merkez
noktasi (cx, cy) uretilir; model KV260 DPU'suna (DPUCZDX8G) derlenir.

**Bagli olmasi gereken Kaggle veri setleri** (Add Input):

| Veri seti | Icerik |
| --- | --- |
| VisDrone2019-DET (train + val) | kara araci: car / van / truck / bus |
| aerial-vehicle-sources | VESSELimg + Military Vehicle Recognition + Mendeley UAV Military |

Ayarlar: **Accelerator = GPU**, **Internet = On**.


In [ ]:
import os
from pathlib import Path

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
%cd {WORK}
!nvidia-smi


In [ ]:
# YOLOX kurulumu (Kaggle'daki hazir torch surumune dokunmadan).
# Kaggle ve Vitis AI VM ayni commit'i kullanir; main dali kullanilmaz.
import importlib
import site
import subprocess
import sys

YOLOX_COMMIT = "6ddff4824372906469a7fae2dc3206c7aa4bbaee"
YOLOX_DIR = Path(WORK) / "YOLOX"

if not YOLOX_DIR.is_dir():
    !git clone --filter=blob:none --no-checkout https://github.com/Megvii-BaseDetection/YOLOX.git "{YOLOX_DIR}"
!git -C "{YOLOX_DIR}" fetch --depth 1 origin {YOLOX_COMMIT}
!git -C "{YOLOX_DIR}" checkout --detach {YOLOX_COMMIT}
current = !git -C "{YOLOX_DIR}" rev-parse HEAD
assert current and current[0] == YOLOX_COMMIT, f"Yanlis YOLOX commit'i: {current}"


def _pip(*args):
    """pip'i cagirir. `!pip` kabuk cagrisinin aksine hatayi yutmaz."""
    proc = subprocess.run([sys.executable, "-m", "pip", *args],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-4000:])
        print(proc.stderr[-4000:])
    return proc.returncode


# --no-build-isolation sart: YOLOX'un setup.py'si torch'u import eder, pip'in
# izole build ortaminda torch bulunmaz ve kurulum sessizce basarisiz olur.
rc = _pip("install", "--no-deps", "--no-build-isolation", "-e", str(YOLOX_DIR))
if rc != 0:
    print("Editable kurulum basarisiz; editable olmayan kuruluma dusuluyor.")
    rc = _pip("install", "--no-deps", "--no-build-isolation", str(YOLOX_DIR))
assert rc == 0, "YOLOX kurulumu basarisiz (yukaridaki pip ciktisina bakin)."

assert _pip("install", "-q", "loguru", "tabulate", "psutil", "pycocotools",
            "thop", "ninja") == 0, "Yardimci paket kurulumu basarisiz."

# pip'in yazdigi .pth dosyalari yalnizca yorumlayici acilisinda okunur; calisan
# kernel'in sys.path'ini elle tazelemezsek import ayni oturumda basarisiz olur.
for _site_dir in getattr(site, "getsitepackages", list)():
    site.addsitedir(_site_dir)
if str(YOLOX_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOX_DIR))
importlib.invalidate_caches()

# numpy 1.24+ uyumlulugu: kaldirilan eski takma adlar icin shim
import numpy as np
for _alias, _type in (("float", float), ("int", int), ("bool", bool)):
    if _alias not in np.__dict__:
        setattr(np, _alias, _type)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
import yolox
print("yolox:", yolox.__version__, "|", yolox.__file__)


In [ ]:
%%writefile build_dataset.py
#!/usr/bin/env python3
"""Dort kaynagi tek bir 2-sinifli COCO veri setinde birlestirir.

Hedef siniflar
--------------
  1 = land_vehicle   2 = sea_vehicle

Kaynaklar ve formatlari
-----------------------
  visdrone   VisDrone-DET yerel txt   (x,y,w,h,score,category,trunc,occ)
  vesselimg  Roboflow COCO JSON
  milrec     Roboflow COCO JSON       (Military Vehicle Recognition)
  mendeley   YOLO txt                 (cls cx cy w h, normalize)

Uc tur karar
------------
  LAND / SEA  -> hedef kutu
  IGNORE      -> iscrowd=1, ignore=1. Ne tespit beklenir ne arka plan sayilir.
  None (at)   -> anotasyon uretilmez; nesne arka plan olur. Hedef disi ve
                 gorsel olarak farkli siniflar icin **istenen** davranis
                 (or. Buoy: "samandira gemi degil" ogretilir).

Bilinen tuzaklar ve alinan onlemler
-----------------------------------
* VESSELimg 23 cekim oturumundan olusuyor ve Roboflow kareleri **rastgele**
  bolmus: 23 oturumun tamami train/valid/test'te birden bulunuyor. Bu haliyle
  dogrulama skoru sahte cikar. Burada bolme **oturum bazli** yeniden yapilir.
* VisDrone'un ignored-region (category 0) ve score=0 crowd semantigi korunur.
* Kutular goruntu sinirlarina kirpilir; sifir/negatif alanlilar atilir.

Kullanim
--------
    python tools/build_dataset.py --data-dir datasets --out datasets/merged
    python tools/build_dataset.py ... --repeat vesselimg=2 --subsample visdrone=0.5
"""

import argparse
import io
import json
import os
import random
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image

TARGET_NAMES = ("land_vehicle", "sea_vehicle")
LAND, SEA = 1, 2
IGNORE = "ignore"

#: VisDrone kategori kimligi -> hedef. 0 = ignored-region (ozel islenir),
#: 11 = others (resmi protokolde hedef degil).
VISDRONE_MAP = {
    1: None,   # pedestrian
    2: None,   # people
    3: None,   # bicycle        - iki tekerlekli, siluet arabaya benzemiyor
    4: LAND,   # car
    5: LAND,   # van
    6: LAND,   # truck
    7: None,   # tricycle
    8: None,   # awning-tricycle
    9: LAND,   # bus
    10: None,  # motor
    11: None,  # others
}

#: Roboflow COCO kategori adi -> hedef. Adlar kucuk harfe indirilerek eslenir.
VESSELIMG_MAP = {
    "container": SEA,
    "chemical": SEA,
    "passenger-roro": SEA,
    "tugboat": SEA,
    "pilot": IGNORE,   # kilavuz botu: deniz araci ama medyan 15 px
    "buoy": None,      # araç degil; kasten hard negative
    "boats": None,     # Roboflow ust-kategori artigi, ornegi yok
}

MILREC_MAP = {
    "tank": LAND,
    "armoured personnel carrier": LAND,
    "air-fighter": None,
    "bomber": None,
    "soldier": None,
    "military-tank-plane-soldier": None,  # Roboflow artigi
}

#: Mendeley data.yaml sirasi: ['tank', 'drone', 'people', 'soldier']
MENDELEY_MAP = {0: LAND, 1: None, 2: None, 3: None}

#: Roboflow kaynaklarinda dogrulamaya ayrilacak oturum orani.
#:
#: Her iki Roboflow export'u da bolmeyi **kare/kopya bazinda** yapmis:
#:   * VESSELimg  - 23 cekim oturumunun tamami train/valid/test'te birden
#:   * milrec     - ayni kaynak goruntunun augment kopyalari farkli bolumlerde
#: Ikisinde de bolme oturum (kaynak goruntu) bazinda yeniden yapilir.
ROBOFLOW_VAL_FRACTION = 0.25
SPLIT_SEED = 1337


# --------------------------------------------------------------------------
# zip veya klasor farkini gizleyen ince katman
# --------------------------------------------------------------------------
class Archive:
    """Bir .zip dosyasini veya bir klasoru ayni arayuzle okur."""

    def __init__(self, path):
        self.path = Path(path)
        self._zip = None
        if self.path.suffix.lower() == ".zip":
            self._zip = zipfile.ZipFile(self.path)
            self._names = [n for n in self._zip.namelist() if not n.endswith("/")]
        elif self.path.is_dir():
            self._names = [
                p.relative_to(self.path).as_posix()
                for p in self.path.rglob("*") if p.is_file()
            ]
        else:
            raise SystemExit(f"HATA: bulunamadi veya desteklenmiyor: {path}")
        self._name_set = set(self._names)

    def names(self):
        return self._names

    def read(self, name):
        if self._zip is not None:
            return self._zip.read(name)
        return (self.path / name).read_bytes()

    def open(self, name):
        if self._zip is not None:
            return self._zip.open(name)
        return open(self.path / name, "rb")

    def exists(self, name):
        return name in self._name_set

    def image_size(self, name):
        """Yalnizca basligi okur; tum goruntuyu cozmez."""
        with self.open(name) as handle:
            with Image.open(io.BytesIO(handle.read()) if self._zip else handle) as im:
                return im.size  # (width, height)

    def close(self):
        if self._zip is not None:
            self._zip.close()
            self._zip = None

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()
        return False


class Record:
    """Birlestirilmis veri setindeki tek bir goruntu."""

    __slots__ = ("source", "member", "file_name", "width", "height",
                 "split", "anns", "ignore_regions", "group")

    def __init__(self, source, member, file_name, width, height, split, group):
        self.source = source          # kaynak adi (visdrone, vesselimg, ...)
        self.member = member          # arsiv icindeki yol
        self.file_name = file_name    # cikti COCO'daki goreli yol
        self.width = width
        self.height = height
        self.split = split            # "train" | "val"
        self.group = group            # sizinti kontrolu icin oturum anahtari
        self.anns = []                # {bbox, category_id, iscrowd, ignore}
        self.ignore_regions = []


def clip_box(x, y, w, h, width, height):
    """Kutuyu goruntuye kirpar; gecersizse None doner."""
    x1 = max(0.0, float(x))
    y1 = max(0.0, float(y))
    x2 = min(float(width), float(x) + float(w))
    y2 = min(float(height), float(y) + float(h))
    if x2 - x1 <= 0 or y2 - y1 <= 0:
        return None
    return [x1, y1, x2 - x1, y2 - y1]


def add_box(record, target, box):
    if target is IGNORE:
        record.anns.append({"bbox": box, "category_id": LAND,
                            "iscrowd": 1, "ignore": 1})
    else:
        record.anns.append({"bbox": box, "category_id": target,
                            "iscrowd": 0, "ignore": 0})


# --------------------------------------------------------------------------
# Kaynak okuyucular
# --------------------------------------------------------------------------
def has_part(name, part):
    """Yol bileseni tam eslesme ile aranir.

    Arsiv kokunun nereye isaret ettigine gore ayni dosya
    'VisDrone2019-DET-train/images/x.jpg' veya 'images/x.jpg' olarak
    gorunebilir; '/images/' arayan bir kontrol ikincisini kaciririrdi.
    """
    return part in name.split("/")


def swap_part(name, old, new):
    parts = name.split("/")
    return "/".join(new if p == old else p for p in parts)


def read_visdrone(archive, split, source="visdrone", stats=None):
    """VisDrone yerel txt formati. category 0 -> ignored-region."""
    names = archive.names()
    images = sorted(n for n in names
                    if has_part(n, "images") and n.lower().endswith(".jpg"))
    records = []
    for member in images:
        stem = member.rsplit("/", 1)[-1].rsplit(".", 1)[0]
        ann_member = swap_part(member, "images", "annotations")
        ann_member = ann_member.rsplit(".", 1)[0] + ".txt"
        if not archive.exists(ann_member):
            continue
        width, height = archive.image_size(member)
        rec = Record(source, member, f"{source}/{stem}.jpg",
                     width, height, split, group=f"{source}:{stem}")
        for line in archive.read(ann_member).decode("utf-8", "replace").splitlines():
            line = line.strip().rstrip(",")
            if not line:
                continue
            parts = line.split(",")
            if len(parts) < 6:
                continue
            try:
                x, y, w, h, score, cat = (int(float(v)) for v in parts[:6])
            except ValueError:
                continue
            box = clip_box(x, y, w, h, width, height)
            if box is None:
                continue
            if cat == 0:
                rec.ignore_regions.append(box)
                continue
            target = VISDRONE_MAP.get(cat, None)
            if target is None:
                if stats is not None:
                    stats[f"{source}:drop:cat{cat}"] += 1
                continue
            if score == 0:            # resmi protokol: sinif-ozel crowd
                add_box(rec, IGNORE, box)
                if stats is not None:
                    stats[f"{source}:ignore:score0"] += 1
            else:
                add_box(rec, target, box)
                if stats is not None:
                    stats[f"{source}:keep:cat{cat}"] += 1
        records.append(rec)
    return records


def read_roboflow_coco(archive, class_map, source, split_override=None,
                       stats=None):
    """Roboflow COCO export: her bolum klasorunde _annotations.coco.json."""
    records = []
    for split_dir in ("train", "valid", "test"):
        member = f"{split_dir}/_annotations.coco.json"
        if not archive.exists(member):
            continue
        data = json.loads(archive.read(member))
        cats = {c["id"]: str(c["name"]).strip().lower()
                for c in data["categories"]}
        unknown = set(cats.values()) - set(class_map)
        if unknown:
            raise SystemExit(
                f"HATA: {source} icinde eslenmemis kategori: {sorted(unknown)}. "
                f"build_dataset.py icindeki esleme tablosunu guncelleyin."
            )
        by_image = defaultdict(list)
        for ann in data["annotations"]:
            by_image[ann["image_id"]].append(ann)

        split = split_override or ("val" if split_dir in ("valid", "test") else "train")
        for image in data["images"]:
            member_img = f"{split_dir}/{image['file_name']}"
            if not archive.exists(member_img):
                raise SystemExit(f"HATA: {source} goruntusu eksik: {member_img}")
            rec = Record(source, member_img, f"{source}/{image['file_name']}",
                         int(image["width"]), int(image["height"]), split,
                         group=session_key(image["file_name"], source))
            for ann in by_image.get(image["id"], ()):
                name = cats[ann["category_id"]]
                target = class_map[name]
                box = clip_box(*ann["bbox"], rec.width, rec.height)
                if box is None:
                    continue
                if target is None:
                    if stats is not None:
                        stats[f"{source}:drop:{name}"] += 1
                    continue
                add_box(rec, target, box)
                key = "ignore" if target is IGNORE else "keep"
                if stats is not None:
                    stats[f"{source}:{key}:{name}"] += 1
            records.append(rec)
    return records


def read_mendeley_yolo(archive, source="mendeley", stats=None):
    """YOLO txt: 'cls cx cy w h', hepsi 0-1 normalize."""
    names = archive.names()
    images = sorted(n for n in names
                    if has_part(n, "images") and n.lower().endswith((".jpg", ".png")))
    records = []
    for member in images:
        label = swap_part(member, "images", "labels").rsplit(".", 1)[0] + ".txt"
        if not archive.exists(label):
            continue
        # .../dataset/<split>/images/<dosya>
        parts = member.split("/")
        split_dir = parts[-3] if len(parts) >= 3 else "train"
        split = "val" if split_dir in ("valid", "val", "test") else "train"
        width, height = archive.image_size(member)
        base = parts[-1]
        rec = Record(source, member, f"{source}/{base}", width, height,
                     split, group=session_key(base, source))
        for line in archive.read(label).decode("utf-8", "replace").splitlines():
            p = line.split()
            if len(p) < 5:
                continue
            try:
                cls = int(float(p[0]))
                cx, cy, bw, bh = (float(v) for v in p[1:5])
            except ValueError:
                continue
            target = MENDELEY_MAP.get(cls, None)
            box = clip_box((cx - bw / 2) * width, (cy - bh / 2) * height,
                           bw * width, bh * height, width, height)
            if box is None:
                continue
            if target is None:
                if stats is not None:
                    stats[f"{source}:drop:cls{cls}"] += 1
                continue
            add_box(rec, target, box)
            if stats is not None:
                stats[f"{source}:keep:cls{cls}"] += 1
        records.append(rec)
    return records


def session_key(file_name, source=""):
    """Ayni cekimden / ayni kaynak goruntuden gelen kareleri gruplayan anahtar.

    Roboflow dosya adlari '<taban>_<kare>_jpg.rf.<hash>.jpg' bicimindedir;
    taban kisim ya cekim oturumunu (kamera + zaman damgasi) ya da augment
    kopyalarinin turedigi kaynak goruntuyu tasir. Ikisi de ayni bolumde
    kalmalidir.

    Anahtar kaynak adiyla oneklenir: iki veri seti de goruntulerini 1, 2, 3
    diye numaraladigi icin onek olmadan alakasiz goruntuler ayni oturum
    sayilirdi. (Kontrol edildi: milrec ve mendeley'deki ayni adli goruntuler
    gorsel olarak tamamen farkli.)
    """
    base = file_name.rsplit("/", 1)[-1]
    prefix = f"{source}:" if source else ""
    match = re.match(r"(.+?)_\d+_(?:jpg|png)\.rf\.", base)
    if match:
        return prefix + match.group(1)
    match = re.match(r"(.+?)\.rf\.", base)
    if match:
        return prefix + match.group(1)
    return prefix + base


def resplit_by_group(records, val_fraction, seed=SPLIT_SEED):
    """Kareleri degil **oturumlari** boler; bitisik kare sizintisini onler."""
    groups = sorted({r.group for r in records})
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_val = max(1, round(len(groups) * val_fraction))
    val_groups = set(groups[:n_val])
    for r in records:
        r.split = "val" if r.group in val_groups else "train"
    return len(groups), len(val_groups)


# --------------------------------------------------------------------------
# Dogrulama kapilari
# --------------------------------------------------------------------------
def validate(records):
    """Bozuk bir sey varsa dosya uretmeden hata verir."""
    problems = []
    seen_files = set()
    groups = defaultdict(set)

    for r in records:
        if r.file_name in seen_files:
            problems.append(f"yinelenen dosya adi: {r.file_name}")
        seen_files.add(r.file_name)
        if r.width <= 0 or r.height <= 0:
            problems.append(f"gecersiz goruntu boyutu: {r.file_name}")
        groups[r.group].add(r.split)
        for ann in r.anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                problems.append(f"sifir/negatif kutu: {r.file_name} {ann['bbox']}")
            if x < -1e-6 or y < -1e-6 or x + w > r.width + 1e-6 \
                    or y + h > r.height + 1e-6:
                problems.append(f"sinir disi kutu: {r.file_name} {ann['bbox']} "
                                f"(goruntu {r.width}x{r.height})")
            if ann["category_id"] not in (LAND, SEA):
                problems.append(f"gecersiz kategori: {ann['category_id']}")

    leaked = [g for g, s in groups.items() if len(s) > 1]
    if leaked:
        problems.append(
            f"{len(leaked)} oturum hem train hem val'de "
            f"(ornek: {sorted(leaked)[:3]})"
        )
    return problems


def build_coco(records, split):
    subset = [r for r in records if r.split == split]
    images, annotations = [], []
    ann_id = 1
    for image_id, rec in enumerate(sorted(subset, key=lambda r: r.file_name), 1):
        images.append({
            "id": image_id,
            "file_name": rec.file_name,
            "width": rec.width,
            "height": rec.height,
            "source": rec.source,
            "ignore_regions": rec.ignore_regions,
        })
        for ann in rec.anns:
            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": ann["category_id"],
                "bbox": [round(v, 2) for v in ann["bbox"]],
                "area": round(ann["bbox"][2] * ann["bbox"][3], 2),
                "iscrowd": ann["iscrowd"],
                "ignore": ann["ignore"],
            })
            ann_id += 1
    return {
        "info": {"description": "Birlesik 2-sinifli havadan arac veri seti",
                 "class_scheme": "land_vehicle+sea_vehicle"},
        "images": images,
        "annotations": annotations,
        "categories": [{"id": i + 1, "name": n, "supercategory": "vehicle"}
                       for i, n in enumerate(TARGET_NAMES)],
    }


def parse_kv(values):
    out = {}
    for item in values or ():
        if "=" not in item:
            raise SystemExit(f"HATA: 'kaynak=deger' bekleniyordu: {item}")
        key, value = item.split("=", 1)
        out[key.strip()] = float(value)
    return out


def summarise(records, stats):
    print("\n" + "=" * 74)
    print("KAYNAK BAZLI OZET")
    print("=" * 74)
    per_source = defaultdict(lambda: Counter())
    for r in records:
        c = per_source[r.source]
        c["goruntu"] += 1
        c[f"goruntu_{r.split}"] += 1
        for a in r.anns:
            if a["ignore"]:
                c["ignore"] += 1
            elif a["category_id"] == LAND:
                c["land"] += 1
            else:
                c["sea"] += 1
        c["ignore_region"] += len(r.ignore_regions)
    header = f"{'kaynak':<12}{'goruntu':>9}{'train':>8}{'val':>7}{'land':>9}{'sea':>8}{'ignore':>8}"
    print(header); print("-" * len(header))
    for source in sorted(per_source):
        c = per_source[source]
        print(f"{source:<12}{c['goruntu']:>9}{c['goruntu_train']:>8}"
              f"{c['goruntu_val']:>7}{c['land']:>9}{c['sea']:>8}{c['ignore']:>8}")

    print("\nSINIF ESLEME DOKUMU (kaynak:karar:sinif)")
    for key in sorted(stats):
        print(f"  {key:<40} {stats[key]:>8}")


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default="datasets",
                        help="zip/klasor kaynaklarinin bulundugu dizin")
    parser.add_argument("--out", default="datasets/merged",
                        help="cikti dizini (instances_train.json / instances_val.json)")
    parser.add_argument("--repeat", nargs="*", default=[],
                        help="or. vesselimg=2 (train'de tekrar sayisi)")
    parser.add_argument("--subsample", nargs="*", default=[],
                        help="or. visdrone=0.5 (train goruntulerinin orani)")
    parser.add_argument("--dry-run", action="store_true",
                        help="dosya yazma, yalnizca dogrula ve raporla")
    parser.add_argument("--images-out", default=None,
                        help="goruntuleri <dizin>/<kaynak>/<ad> duzeninde cikar "
                             "(COCO file_name alanlariyla birebir ortusur)")
    parser.add_argument("--source", nargs="*", default=[], metavar="ETIKET=YOL",
                        help="tek bir kaynagin yolunu degistir. Kaggle'da her "
                             "veri seti ayri bir /kaggle/input klasorunde "
                             "oldugu icin gerekir. Etiketler: visdrone-train, "
                             "visdrone-val, vesselimg, milrec, mendeley")
    args = parser.parse_args()

    data_dir = Path(args.data_dir)
    stats = Counter()
    records = []

    plan = [
        ("visdrone-train", "VisDrone2019-DET-train.zip", "visdrone", "train"),
        ("visdrone-val", "VisDrone2019-DET-val.zip", "visdrone", "val"),
        ("vesselimg", "VESSELimg.v4i.coco.zip", "vesselimg", None),
        ("milrec", "Military Vehicle Recognition.v7i.coco.zip", "milrec", None),
        ("mendeley", "A Multi-Class UAV Military Object Detection Datase.zip",
         "mendeley", None),
    ]

    overrides = {}
    for item in args.source:
        if "=" not in item:
            raise SystemExit(f"HATA: 'etiket=yol' bekleniyordu: {item}")
        key, value = item.split("=", 1)
        overrides[key.strip()] = Path(value.strip())
    unknown = set(overrides) - {label for label, _, _, _ in plan}
    if unknown:
        raise SystemExit(f"HATA: bilinmeyen kaynak etiketi: {sorted(unknown)}")

    resolved = {}
    for label, filename, source, split in plan:
        if label in overrides:
            path = overrides[label]
            if not path.exists():
                raise SystemExit(f"HATA: --source {label} yolu yok: {path}")
        else:
            path = data_dir / filename
            if not path.exists():
                alt = data_dir / Path(filename).stem
                if alt.exists():
                    path = alt
                else:
                    raise SystemExit(f"HATA: kaynak bulunamadi: {path}")
        resolved[label] = path

    for label, filename, source, split in plan:
        path = resolved[label]
        print(f">> {label:<16} {path.name}")
        archive = Archive(path)
        try:
            if source == "visdrone":
                got = read_visdrone(archive, split, stats=stats)
            elif source in ("vesselimg", "milrec"):
                class_map = VESSELIMG_MAP if source == "vesselimg" else MILREC_MAP
                got = read_roboflow_coco(archive, class_map, source, stats=stats)
                # Roboflow'un kendi bolmesi guvenilmez (bkz. ROBOFLOW_VAL_FRACTION)
                total, val = resplit_by_group(got, ROBOFLOW_VAL_FRACTION)
                print(f"   oturum bazli yeniden bolme: {total} oturum -> {val} val")
            else:
                got = read_mendeley_yolo(archive, stats=stats)
                # Mendeley de Roboflow'dan gecmis (.rf. hash'i); ayni kaynak
                # goruntunun augment kopyalari train/test'e dagilmis durumda.
                total, val = resplit_by_group(got, ROBOFLOW_VAL_FRACTION)
                print(f"   oturum bazli yeniden bolme: {total} oturum -> {val} val")
        finally:
            archive.close()
        if not got:
            raise SystemExit(
                f"HATA: '{label}' kaynagindan hic goruntu okunamadi ({path}).\n"
                f"  Olasi sebep: klasor duzeni beklenenden farkli, ya da\n"
                f"  VisDrone icin 'annotations/' yerine YOLO 'labels/' var.\n"
                f"  Ultralytics'e cevrilmis VisDrone kopyalari ignored-region\n"
                f"  ve score=0 crowd bilgisini siler; bu boru hatti orijinal\n"
                f"  VisDrone txt formatini bekler."
            )
        print(f"   {len(got)} goruntu")
        records.extend(got)

    # --- tekrar / seyreltme (yalnizca train) ---
    repeat = parse_kv(args.repeat)
    subsample = parse_kv(args.subsample)
    if subsample:
        rng = random.Random(SPLIT_SEED)
        kept = []
        for r in records:
            frac = subsample.get(r.source)
            if r.split == "train" and frac is not None and rng.random() > frac:
                continue
            kept.append(r)
        print(f"\nseyreltme sonrasi: {len(records)} -> {len(kept)} goruntu")
        records = kept
    if repeat:
        extra = []
        for r in records:
            times = int(repeat.get(r.source, 1))
            if r.split == "train" and times > 1:
                extra.extend([r] * (times - 1))
        if extra:
            print(f"tekrar sonrasi: +{len(extra)} goruntu girisi")
            records.extend(extra)

    problems = validate([r for r in records if True])
    if problems:
        print("\n" + "!" * 74)
        print(f"DOGRULAMA BASARISIZ - {len(problems)} sorun")
        for p in problems[:20]:
            print("  -", p)
        if len(problems) > 20:
            print(f"  ... ve {len(problems)-20} tane daha")
        raise SystemExit(1)
    print("\nDogrulama gecti: kutular sinir icinde, kategori {1,2}, "
          "oturum sizintisi yok.")

    summarise(records, stats)

    train = build_coco(records, "train")
    val = build_coco(records, "val")
    print("\n" + "=" * 74)
    print(f"{'':<8}{'goruntu':>10}{'kutu':>10}{'land':>9}{'sea':>8}{'ignore':>8}")
    for name, coco in (("train", train), ("val", val)):
        real = [a for a in coco["annotations"] if not a["ignore"]]
        land = sum(1 for a in real if a["category_id"] == LAND)
        sea = sum(1 for a in real if a["category_id"] == SEA)
        ign = len(coco["annotations"]) - len(real)
        print(f"{name:<8}{len(coco['images']):>10}{len(real):>10}"
              f"{land:>9}{sea:>8}{ign:>8}")
        if sea and land:
            print(f"{'':8}dengesizlik land:sea = {land/sea:.1f} : 1")

    if args.dry_run:
        print("\n--dry-run: dosya yazilmadi.")
        return

    out = Path(args.out)
    (out / "annotations").mkdir(parents=True, exist_ok=True)
    for name, coco in (("train", train), ("val", val)):
        target = out / "annotations" / f"instances_{name}.json"
        target.write_text(json.dumps(coco), encoding="utf-8")
        print(f"yazildi: {target}")

    manifest = out / "image_manifest.json"
    manifest.write_text(json.dumps({
        "sources": {label: filename for label, filename, _, _ in plan},
        "members": [{"source": r.source, "member": r.member,
                     "file_name": r.file_name} for r in records],
    }), encoding="utf-8")
    print(f"yazildi: {manifest}  (goruntuleri cikarmak icin)")

    if args.images_out:
        extract_images(records, Path(args.images_out), resolved, plan)


def extract_images(records, images_out, resolved, plan):
    """Goruntuleri COCO `file_name` alanlariyla ortusen duzene cikarir.

    Sonuc: <images_out>/<kaynak>/<ad>.jpg  ->  YOLOX tarafinda
    data_dir=<ust dizin>, name="images" ile dogrudan okunur.
    Var olan dosyalar atlanir; yarim kalan cikarma guvenle tekrarlanabilir.
    """
    source_archive = {}
    for label, _, source, _ in plan:
        source_archive.setdefault(source, resolved[label])

    # Ayni kayit tekrar katsayisi yuzunden birden fazla kez gelebilir.
    unique = {r.file_name: r for r in records}
    written = skipped = 0
    for source in sorted({r.source for r in unique.values()}):
        with Archive(source_archive[source]) as archive:
            for rec in sorted((r for r in unique.values() if r.source == source),
                              key=lambda r: r.file_name):
                target = images_out / rec.file_name
                if target.exists() and target.stat().st_size > 0:
                    skipped += 1
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(rec.member))
                written += 1
        print(f"   {source}: cikarildi")
    print(f"goruntuler: {written} yazildi, {skipped} zaten vardi -> {images_out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile visdrone_eval.py
#!/usr/bin/env python3
"""VisDrone DET toolkit'inin AP@500 protokolunun saf Python karsiligi.

AP hesabi resmi `calcAccuracy.m` ile birebir ayni: ignore GT'ler recall
paydasinda kalir (`rec = tp/max(1,numel(gtMatch))`).

Ek olarak, resmi protokolde bulunmayan iki tamamlayici cikti uretilir:
  * P/R/F1 (IoU 0.50) - sabit esikte ve en iyi F1 noktasinda. Yayinlanmis
    YOLO calismalariyla kiyaslanabilmesi icin bu metrigin recall paydasi
    ignore GT'leri **icermez**; AP'ninkinden farklidir, bilincli.
  * Sinif gruplama - 10 ince sinifi 3 saha sinifina indirger. Eslestirme
    hem GT'ye hem tespitlere uygulanir, yani `van`-`car` karisikligi
    degerlendirme asamasinda ortadan kalkar; model yeniden egitilmez.
"""

from collections import defaultdict

import numpy as np

VISDRONE_CLASSES = (
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor",
)

#: 10 ince VisDrone sinifi -> 3 saha sinifi. Sahada ayrilmasi anlamli olan
#: gruplar korunur; 20 pikselde insanin bile ayiramadigi ayrimlar birlesir.
GROUP_3 = {
    1: 1, 2: 1,                  # pedestrian, people      -> person
    3: 2, 7: 2, 8: 2, 10: 2,     # bicycle, tricycle,
                                 # awning-tricycle, motor  -> two/three-wheeler
    4: 3, 5: 3, 6: 3, 9: 3,      # car, van, truck, bus    -> vehicle
}
GROUP_3_NAMES = {1: "person", 2: "twowheeler", 3: "vehicle"}

#: Hedef saha tanimi: person + vehicle. `vehicle` her turlu tasiti kapsar
#: (bisiklet ve motosiklet dahil). Egitim de bu tanimla yapilir.
GROUP_2 = {
    1: 1, 2: 1,                                   # pedestrian, people -> person
    3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2,   # geri kalani -> vehicle
}
GROUP_2_NAMES = {1: "person", 2: "vehicle"}

#: Tek sinifli tanimlar. Eslenmeyen kategoriler hem GT'den hem tespitlerden
#: **dusurulur** (gruplanmaz): sistem o nesneleri hedeflemiyorsa onlari
#: bulamamak da bulmak da puanlanmamalidir.
PERSON_ONLY = {1: 1, 2: 1}
VEHICLE_ONLY = {k: 1 for k in (3, 4, 5, 6, 7, 8, 9, 10)}

#: Degerlendirme senaryolari: ad -> (kategori eslemesi, sinif adlari).
#: `None` esleme resmi 10 sinifli protokol demektir.
SCENARIOS = {
    "10 sinif (resmi)": (None, None),
    "2 sinif (hedef)": (GROUP_2, GROUP_2_NAMES),
    "3 sinif (ara)": (GROUP_3, GROUP_3_NAMES),
    "tek sinif: person": (PERSON_ONLY, {1: "person"}),
    "tek sinif: vehicle": (VEHICLE_ONLY, {1: "vehicle"}),
}

#: En iyi F1 aramasinda taranan guven esikleri.
SCORE_GRID = np.round(np.arange(0.0, 1.0001, 0.01), 4)


def covered_fraction_xywh(box, regions):
    """Bir xywh kutusunun ignore dikdortgenleri birlesimi icindeki oranini bulur."""
    x, y, w, h = (float(v) for v in box)
    if w <= 0 or h <= 0:
        return 0.0
    clipped = []
    for rx, ry, rw, rh in regions:
        left, top = max(x, rx), max(y, ry)
        right, bottom = min(x + w, rx + rw), min(y + h, ry + rh)
        if right > left and bottom > top:
            clipped.append((left, top, right, bottom))
    if not clipped:
        return 0.0

    # Kesisen dikdortgenlerin birlesim alanini x-sweep ile cift saymadan hesapla.
    xs = sorted({edge for rect in clipped for edge in (rect[0], rect[2])})
    covered = 0.0
    for left, right in zip(xs, xs[1:]):
        if right <= left:
            continue
        intervals = sorted(
            (top, bottom)
            for x1, top, x2, bottom in clipped
            if x1 < right and x2 > left
        )
        union_y = 0.0
        if intervals:
            start, end = intervals[0]
            for next_start, next_end in intervals[1:]:
                if next_start > end:
                    union_y += end - start
                    start, end = next_start, next_end
                else:
                    end = max(end, next_end)
            union_y += end - start
        covered += (right - left) * union_y
    return covered / (w * h)


def _overlap(dt, gt, gt_ignore):
    dx, dy, dw, dh = dt
    gx, gy, gw, gh = gt
    iw = min(dx + dw, gx + gw) - max(dx, gx)
    ih = min(dy + dh, gy + gh) - max(dy, gy)
    if iw <= 0 or ih <= 0:
        return 0.0
    inter = iw * ih
    if gt_ignore:
        return inter / max(dw * dh, 1e-12)
    union = dw * dh + gw * gh - inter
    return inter / max(union, 1e-12)


def _eval_image(gt_rows, detections, threshold):
    """VisDrone toolkit evalRes.m ile ayni eslestirme sirasi."""
    # Normal GT once gelir; ignore GT en sona gelir ve birden cok kez eslesebilir.
    gt_rows = sorted(gt_rows, key=lambda row: bool(row["ignore"]))
    gt_state = [-1 if row["ignore"] else 0 for row in gt_rows]
    detections = sorted(detections, key=lambda row: -row["score"])
    dt_matches = []

    for detection in detections:
        best_overlap = threshold
        best_index = None
        best_match = 0
        for index, (ground_truth, state) in enumerate(zip(gt_rows, gt_state)):
            if state == 1:
                continue
            if best_match != 0 and state == -1:
                break
            overlap = _overlap(
                detection["bbox"], ground_truth["bbox"], state == -1
            )
            if overlap < best_overlap:
                continue
            best_overlap = overlap
            best_index = index
            best_match = 1 if state == 0 else -1
        if best_index is not None and best_match == 1:
            gt_state[best_index] = 1
        dt_matches.append((detection["score"], best_match))
    return gt_state, dt_matches


def _voc_ap(recall, precision):
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))
    for index in range(precision.size - 2, -1, -1):
        precision[index] = max(precision[index], precision[index + 1])
    changes = np.where(recall[1:] != recall[:-1])[0] + 1
    return float(
        np.sum((recall[changes] - recall[changes - 1]) * precision[changes])
    )


def prepare_detections(coco, detections, max_dets=500):
    """Global ignore filtresi ve goruntu basina global top-500 uygular."""
    grouped = defaultdict(list)
    for detection in detections:
        image_id = int(detection["image_id"])
        regions = coco.imgs[image_id].get("ignore_regions", ())
        if covered_fraction_xywh(detection["bbox"], regions) >= 0.5:
            continue
        grouped[image_id].append(detection)

    prepared = {}
    for image_id in coco.getImgIds():
        rows = grouped.get(image_id, ())
        prepared[image_id] = sorted(
            rows,
            key=lambda row: (
                -float(row["score"]),
                int(row["category_id"]),
                tuple(float(v) for v in row["bbox"]),
            ),
        )[:max_dets]
    return prepared


def _f1_curve(dt_rows, real_gt_count):
    """Skor esigi izgarasi uzerinde precision/recall/F1 dizileri uretir.

    `dt_rows` skora gore azalan sirali (skor, eslesme) ciftleridir; eslesme
    1=TP, 0=FP, -1=ignore GT ile eslesti (ne TP ne FP sayilir).

    Recall paydasi **ignore olmayan** GT sayisidir. AP'nin paydasindan
    (resmi toolkit ignore'lari da sayar) bilincli olarak farklidir: bu metrik
    yayinlanmis YOLO sonuclariyla kiyaslanabilsin diye COCO/Ultralytics
    kuralini izler.
    """
    zeros = np.zeros_like(SCORE_GRID)
    if real_gt_count <= 0:
        return zeros, zeros, zeros

    scores = np.asarray([row[0] for row in dt_rows], dtype=np.float64)
    matches = np.asarray([row[1] for row in dt_rows], dtype=np.int8)
    # scores azalan sirali -> -scores artan; "skor >= t" sayisi searchsorted ile
    kept = np.searchsorted(-scores, -SCORE_GRID, side="right")
    tp_cum = np.concatenate(([0.0], np.cumsum(matches == 1, dtype=np.float64)))
    fp_cum = np.concatenate(([0.0], np.cumsum(matches == 0, dtype=np.float64)))
    tp, fp = tp_cum[kept], fp_cum[kept]

    precision = tp / np.maximum(1e-12, tp + fp)
    recall = tp / float(real_gt_count)
    f1 = 2 * precision * recall / np.maximum(1e-12, precision + recall)
    return precision, recall, f1


def _at_grid(index, precision, recall, f1):
    return {
        "precision": float(precision[index]),
        "recall": float(recall[index]),
        "f1": float(f1[index]),
        "score": float(SCORE_GRID[index]),
    }


def names_from_coco(coco, fallback_count=10):
    """Sinif adlarini COCO kategorilerinden okur.

    Boylece veri semasi degistiginde (10 sinif -> 2 sinif) tablolar
    kendiliginden dogru kalir; ad listesi ikinci bir yerde tekrarlanmaz.
    """
    cats = getattr(coco, "cats", None)
    if cats:
        return {int(k): str(v.get("name", k)) for k, v in cats.items()}
    return {i + 1: n for i, n in enumerate(VISDRONE_CLASSES[:fallback_count])}


def evaluate_visdrone(coco, detections, max_dets=500, group_map=None,
                      class_names=None, score_thr=0.30, num_classes=None):
    """AP@[.50:.95], AP50, AP75 ve tamamlayici P/R/F1 degerlerini hesaplar.

    `group_map` verilirse (ornegin `GROUP_3`), sinif kimlikleri hem GT'de hem
    tespitlerde eslenerek degerlendirme gruplanmis siniflar uzerinde yapilir.
    Model degismez; yalnizca olcum degisir.
    """
    prepared = prepare_detections(coco, detections, max_dets=max_dets)
    if num_classes is None:
        num_classes = len(getattr(coco, "cats", None) or VISDRONE_CLASSES)

    def mapped(category_id):
        if group_map is None:
            return category_id if 1 <= category_id <= num_classes else None
        return group_map.get(category_id)

    gt_by_image_class = defaultdict(list)
    dt_by_image_class = defaultdict(list)
    real_gt_count = defaultdict(int)
    available_classes = set()

    for image_id in coco.getImgIds():
        regions = coco.imgs[image_id].get("ignore_regions", ())
        for annotation in coco.imgToAnns.get(image_id, ()):
            category_id = mapped(int(annotation["category_id"]))
            if category_id is None:
                continue
            if covered_fraction_xywh(annotation["bbox"], regions) >= 0.5:
                continue
            ignore = bool(
                annotation.get("ignore", 0) or annotation.get("iscrowd", 0)
            )
            gt_by_image_class[(image_id, category_id)].append({
                "bbox": [float(v) for v in annotation["bbox"]],
                "ignore": ignore,
            })
            if not ignore:
                real_gt_count[category_id] += 1
            available_classes.add(category_id)
        # Tespitleri sinifa gore bir kez ayir: esik dongusunde tekrarlanmasin.
        for row in prepared[image_id]:
            category_id = mapped(int(row["category_id"]))
            if category_id is not None:
                dt_by_image_class[(image_id, category_id)].append(row)

    thresholds = np.arange(0.50, 0.951, 0.05)
    image_ids = coco.getImgIds()
    per_class = {}
    iou50_matches = {}
    for category_id in sorted(available_classes):
        aps = []
        for index, threshold in enumerate(thresholds):
            gt_matches = []
            dt_matches = []
            for image_id in image_ids:
                gt_state, image_dt = _eval_image(
                    gt_by_image_class.get((image_id, category_id), ()),
                    dt_by_image_class.get((image_id, category_id), ()),
                    threshold,
                )
                gt_matches.extend(gt_state)
                dt_matches.extend(image_dt)

            dt_matches.sort(key=lambda row: -row[0])
            if index == 0:  # IoU 0.50 -> F1 egrisi buradan cikar
                iou50_matches[category_id] = dt_matches
            matches = np.asarray([row[1] for row in dt_matches], dtype=np.int8)
            tp = np.cumsum(matches == 1, dtype=np.float64)
            fp = np.cumsum(matches == 0, dtype=np.float64)
            recall = tp / max(1, len(gt_matches))
            precision = tp / np.maximum(1.0, tp + fp)
            aps.append(_voc_ap(recall, precision))
        per_class[category_id] = aps

    names = dict(class_names or {})
    if not names:
        if group_map is None:
            names = names_from_coco(coco, num_classes)
        elif group_map == GROUP_2:
            names = dict(GROUP_2_NAMES)
        elif group_map == GROUP_3:
            names = dict(GROUP_3_NAMES)
        else:
            # Bilinmeyen esleme: adi kaynak sinif adlarindan turet ki tablo
            # "grup 1" gibi anlamsiz bir etiket gostermesin.
            members = defaultdict(list)
            for source, target in sorted(group_map.items()):
                members[target].append(VISDRONE_CLASSES[source - 1])
            names = {t: "+".join(v) for t, v in members.items()}

    empty = {
        "ap": 0.0, "ap50": 0.0, "ap75": 0.0, "per_class": {},
        "per_class_ap50": {}, "class_names": names,
        "f1_best": None, "f1_at": None, "per_class_f1_best": {},
    }
    if not per_class:
        return empty

    curves = {
        category_id: _f1_curve(
            iou50_matches[category_id], real_gt_count[category_id]
        )
        for category_id in sorted(per_class)
    }
    precision_matrix = np.asarray([c[0] for c in curves.values()])
    recall_matrix = np.asarray([c[1] for c in curves.values()])
    f1_matrix = np.asarray([c[2] for c in curves.values()])
    macro = (
        precision_matrix.mean(axis=0),
        recall_matrix.mean(axis=0),
        f1_matrix.mean(axis=0),
    )
    best_index = int(np.argmax(macro[2]))
    fixed_index = int(np.argmin(np.abs(SCORE_GRID - score_thr)))

    matrix = np.asarray(list(per_class.values()), dtype=np.float64)
    return {
        "ap": float(matrix.mean()),
        "ap50": float(matrix[:, 0].mean()),
        "ap75": float(matrix[:, 5].mean()),
        "per_class": {
            category_id: float(np.mean(values))
            for category_id, values in per_class.items()
        },
        "per_class_ap50": {
            category_id: float(values[0])
            for category_id, values in per_class.items()
        },
        "class_names": names,
        "f1_best": _at_grid(best_index, *macro),
        "f1_at": _at_grid(fixed_index, *macro),
        "per_class_f1_best": {
            category_id: _at_grid(int(np.argmax(curve[2])), *curve)
            for category_id, curve in curves.items()
        },
    }


def evaluate_scenarios(coco, detections, max_dets=500, score_thr=0.30,
                       scenarios=None):
    """Ayni tespitleri birden cok sinif tanimiyla degerlendirir.

    Model bir kez calisir, olcum dort farkli sekilde yapilir. Boylece "tek
    sinifa inersem ne kazanirim" sorusu yeniden egitim yapmadan cevaplanir.
    """
    results = {}
    for name, (group_map, class_names) in (scenarios or SCENARIOS).items():
        results[name] = evaluate_visdrone(
            coco, detections, max_dets=max_dets, group_map=group_map,
            class_names=class_names, score_thr=score_thr,
        )
    return results


def format_scenarios(results):
    """Senaryo karsilastirma tablosunu tek metne cevirir."""
    header = (
        f"{'senaryo':<22}{'AP':>8}{'AP50':>8}{'AP75':>8}"
        f"{'F1':>8}{'P':>8}{'R':>8}{'@conf':>7}"
    )
    lines = [header, "-" * len(header)]
    for name, metrics in results.items():
        best = metrics.get("f1_best")
        if best is None:
            lines.append(f"{name:<22}{'tespit yok':>47}")
            continue
        lines.append(
            f"{name:<22}{metrics['ap']:>8.4f}{metrics['ap50']:>8.4f}"
            f"{metrics['ap75']:>8.4f}{best['f1']:>8.4f}"
            f"{best['precision']:>8.4f}{best['recall']:>8.4f}"
            f"{best['score']:>7.2f}"
        )
    return "\n".join(lines)


def format_metrics(metrics, title="VisDrone"):
    """Degerlendirme ciktisini insan okunur tek bir metne cevirir."""
    names = metrics.get("class_names", {})
    lines = [
        f"{title}: AP@[.50:.95]={metrics['ap']:.4f}  "
        f"AP@0.50={metrics['ap50']:.4f}  AP@0.75={metrics['ap75']:.4f}",
    ]
    best, fixed = metrics.get("f1_best"), metrics.get("f1_at")
    if best and fixed:
        lines.append(
            f"  En iyi F1={best['f1']:.4f} (P={best['precision']:.4f} "
            f"R={best['recall']:.4f} @conf={best['score']:.2f})"
        )
        lines.append(
            f"  conf={fixed['score']:.2f}: F1={fixed['f1']:.4f} "
            f"P={fixed['precision']:.4f} R={fixed['recall']:.4f}"
        )
    if metrics.get("per_class"):
        lines.append(f"  {'sinif':<18}{'AP':>8}{'AP50':>8}{'F1':>8}")
        for category_id in sorted(metrics["per_class"]):
            f1 = metrics["per_class_f1_best"].get(category_id, {})
            lines.append(
                f"  {names.get(category_id, category_id):<18}"
                f"{metrics['per_class'][category_id]:>8.4f}"
                f"{metrics['per_class_ap50'][category_id]:>8.4f}"
                f"{f1.get('f1', float('nan')):>8.4f}"
            )
    return "\n".join(lines)


In [ ]:
%%writefile yolox_nano_visdrone.py
#!/usr/bin/env python3
"""KV260 DPU'suna (DPUCZDX8G) uyumlu YOLOX-Nano deneyi.

Veri: dort kaynaktan birlestirilmis 2 sinifli havadan arac seti
(bkz. tools/build_dataset.py). Dosya adi tarihsel sebeple korunmustur.

DPU uyumlulugu icin iki degisiklik yapilir:
  1. act="relu": DPU, YOLOX'un varsayilan SiLU aktivasyonunu desteklemez.
  2. Focus stem'i DPUFocus ile degistirilir: Focus'un strided-slice islemi
     DPU'da calismaz; sabit one-hot agirlikli 2x2/stride-2 conv ayni
     space-to-depth dizilimini birebir uretir. Boylece model tek DPU
     subgraph'i olarak derlenir ve onceden egitilmis agirliklar gecerli kalir.
"""

import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

for _alias, _type in (("float", float), ("int", int), ("bool", bool)):
    if _alias not in np.__dict__:
        setattr(np, _alias, _type)

from yolox.data import COCODataset
from yolox.evaluators import COCOEvaluator
from yolox.exp import Exp as MyExp
from yolox.utils import is_main_process

for _helper_dir in (Path(__file__).resolve().parent,
                    Path(__file__).resolve().parent.parent):
    if str(_helper_dir) not in sys.path:
        sys.path.insert(0, str(_helper_dir))
from visdrone_eval import evaluate_visdrone, format_metrics  # noqa: E402

#: Saha tanimi: havadan bakista kara ve deniz araclari.
#:   land_vehicle - VisDrone car/van/truck/bus + askeri tank/ZPT
#:   sea_vehicle  - VESSELimg container/chemical/ro-ro/tugboat
#: Veri `tools/build_dataset.py` ile dort kaynaktan birlestirilir. Sinif
#: adlari degerlendirmede COCO kategorilerinden okundugu icin tablolar
#: sema degisse de dogru kalir.
TARGET_CLASSES = ("land_vehicle", "sea_vehicle")


def assert_class_scheme(coco, expected):
    """Anotasyon semasi modelin sinif sayisiyla uyusmuyorsa hemen durur.

    Eski bir sema ile uretilmis `instances_*.json` diskte kalirsa model
    sessizce yanlis etiketlerle egitilir ve bu ancak saatler sonra
    degerlendirmede fark edilir. Ucuz bir kapi, pahali bir hatayi onler.
    """
    if expected is None:
        return
    found = sorted(coco.cats)
    if found != list(range(1, expected + 1)):
        names = [coco.cats[c].get("name", c) for c in found]
        raise ValueError(
            f"Anotasyon semasi uyusmuyor: {expected} sinif bekleniyordu, "
            f"{len(found)} bulundu ({names}). Veriyi "
            f"'tools/build_dataset.py' ile yeniden uretin."
        )


class VisDroneTrainingDataset(COCODataset):
    """Egitimde bilincli olarak ignore edilen alanlari negatif ornek yapmaz.

    Donusturucu genel bolgeleri image.ignore_regions, score=0 kutularini ise
    ``iscrowd=1`` olarak saklar. YOLOX crowd kutularini hedeflerden cikartir;
    burada ek olarak ilgili pikseller 114 ile maskelenir.
    """

    def __init__(self, *args, expected_num_classes=None, **kwargs):
        if kwargs.get("cache", False):
            raise ValueError("Ignore maskeleme ile --cache kullanmayin.")
        super().__init__(*args, **kwargs)
        assert_class_scheme(self.coco, expected_num_classes)
        self._collect_ignore_boxes()

    def _collect_ignore_boxes(self):
        self.ignore_boxes = {}
        for image_id in self.ids:
            ann_ids = self.coco.getAnnIds(imgIds=[int(image_id)], iscrowd=True)
            boxes = {
                tuple(int(round(v)) for v in ann["bbox"])
                for ann in self.coco.loadAnns(ann_ids)
            }
            boxes.update(
                tuple(int(round(v)) for v in bbox)
                for bbox in self.coco.imgs[int(image_id)].get("ignore_regions", ())
            )
            self.ignore_boxes[int(image_id)] = sorted(boxes)

    def load_image(self, index):
        img = super().load_image(index)
        height, width = img.shape[:2]
        for x, y, w, h in self.ignore_boxes.get(int(self.ids[index]), ()):
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + w), min(height, y + h)
            if x2 > x1 and y2 > y1:
                img[y1:y2, x1:x2] = 114
        return img


class VisDroneEvaluator(COCOEvaluator):
    """Resmi DET toolkit eslestirme/VOC AP mantigiyla VisDrone AP@500."""

    def evaluate_prediction(self, data_dict, statistics):
        if not is_main_process():
            return 0, 0, None
        if not data_dict:
            return 0, 0, "VisDrone AP@500: hic tespit yok\n"

        coco_gt = self.dataloader.dataset.coco
        # Sinif adlari COCO kategorilerinden okunur: veri semasi degisirse
        # tablo da kendiliginden dogru kalir.
        metrics = evaluate_visdrone(
            coco_gt, data_dict, max_dets=500,
            num_classes=self.num_classes, score_thr=self.deploy_conf,
        )
        ap, ap50 = metrics["ap"], metrics["ap50"]

        inference_time, nms_time, n_samples = (v.item() for v in statistics)
        batch = self.dataloader.batch_size
        info = (
            "VisDrone DET-style evaluation (ignore, global maxDets=500)\n"
            + format_metrics(metrics, "saha") + "\n"
            + f"Average forward = {1000 * inference_time / (n_samples * batch):.2f} ms, "
            + f"NMS = {1000 * nms_time / (n_samples * batch):.2f} ms\n"
        )
        return ap, ap50, info


class DPUFocus(nn.Module):
    """YOLOX Focus katmaninin DPU-uyumlu birebir karsiligi.

    Focus'un dilimleme + birlestirme sirasi (TL, BL, TR, BR) sabit agirlikli
    bir conv ile ayni matematikle uretilir. `conv` alt modul adi korundugu
    icin onceden egitilmis Focus agirliklari dogrudan yuklenebilir.
    """

    def __init__(self, in_channels, out_channels, ksize=1, stride=1, act="silu"):
        super().__init__()
        from yolox.models.network_blocks import BaseConv

        self.space_to_depth = nn.Conv2d(
            in_channels, in_channels * 4, kernel_size=2, stride=2, bias=False
        )
        w = torch.zeros(in_channels * 4, in_channels, 2, 2)
        # Focus birlestirme sirasi: TL=(0,0), BL=(1,0), TR=(0,1), BR=(1,1)
        for block, (r, c) in enumerate(((0, 0), (1, 0), (0, 1), (1, 1))):
            for ch in range(in_channels):
                w[block * in_channels + ch, ch, r, c] = 1.0
        with torch.no_grad():
            self.space_to_depth.weight.copy_(w)
        self.space_to_depth.weight.requires_grad = False
        self.conv = BaseConv(in_channels * 4, out_channels, ksize, stride, act=act)

    def forward(self, x):
        return self.conv(self.space_to_depth(x))


class Exp(MyExp):
    def __init__(self):
        super().__init__()
        # ---- model: YOLOX-Nano geometrisi, DPU-uyumlu aktivasyon ----
        self.depth = 0.33
        self.width = 0.25
        self.act = "relu"
        self.num_classes = len(TARGET_CLASSES)

        # ---- girdi boyutu: kaynak videonun en-boy oranina uydurulmus ----
        # 1920x1080'i 640x640'a letterbox etmek kanvasin %44'unu gri dolguya
        # harcar ve etkin olcek 0.333'te kalir. 896x512 (16:9) ayni kareyi
        # 0.467 olcekle isler: %12 daha fazla hesapla %40 daha yuksek
        # cozunurluk. Kucuk nesne recall'unun ana kaldiraci budur.
        # (h, w) sirasi YOLOX kuralidir.
        self.input_size = (512, 896)
        self.test_size = (512, 896)
        # random_resize yuksekligi 32*s, genisligi 32*int(s*896/512) yapar;
        # s in [14, 18] -> 448x768 .. 576x992 araligi.
        self.random_size = (14, 18)

        # ---- veri seti ----
        # build_dataset.py ciktisi: annotations/ + images/<kaynak>/<ad>.jpg
        # COCO file_name alanlari kaynak alt klasorunu tasidigi icin hem
        # egitim hem dogrulama ayni "images" kokunu kullanir.
        self.data_dir = "datasets/merged"
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"
        self.image_folder = "images"
        self.data_num_workers = 2
        self.dataset = None

        # ---- augmentasyon (nano varsayilanlari) ----
        self.mosaic_prob = 0.5
        self.mosaic_scale = (0.5, 1.5)
        self.enable_mixup = False

        # ---- egitim ----
        # 10 sinifli calistirmada model epoch ~40'ta doymustu (epoch 45 -> 80
        # arasi AP@0.50 yalnizca +0.006). 40 epoch ayni sonucu yari surede
        # verir. Son 10 epoch mosaic kapali + L1 loss acik: kutu hassasiyetini
        # iyilestirdigi icin (asil darbogazimiz) bu faz korunuyor.
        self.max_epoch = 40
        self.warmup_epochs = 5
        self.no_aug_epochs = 10
        self.eval_interval = 5

        # ---- dagitim calisma noktasi ----
        # Kartta dusuk esikle calisip false positive'leri tracker'in "N karede
        # gorunmeli" kurali ile eleyecegiz (ByteTrack mantigi). Degerlendirme
        # F1'i bu esikte de raporlar.
        self.deploy_conf = 0.15
        self.print_interval = 50
        self.save_history_ckpt = False  # Kaggle diskini doldurmamak icin

        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

            in_channels = [256, 512, 1024]
            # Nano: depthwise=True
            backbone = YOLOPAFPN(
                self.depth, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            head = YOLOXHead(
                self.num_classes, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            self.model = YOLOX(backbone, head)
            # DPU: slice tabanli Focus stem'i esdeger conv surumuyle degistir
            self.model.backbone.backbone.stem = DPUFocus(
                3, int(self.width * 64), ksize=3, act=self.act
            )

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model

    def get_dataset(self, cache=False, cache_type="ram"):
        from yolox.data import TrainTransform

        return VisDroneTrainingDataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            name=self.image_folder,
            img_size=self.input_size,
            expected_num_classes=self.num_classes,
            # VisDrone karelerinde yuzlerce nesne olabilir; varsayilan 50 cok dusuk
            preproc=TrainTransform(
                max_labels=1000, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob
            ),
            cache=cache,
            cache_type=cache_type,
        )

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        # yolox_base.Exp.get_data_loader kopyasi; tek fark mozaik donusumunde
        # max_labels=4000 (4 yogun VisDrone karesinin mozaigini kirpmaz).
        import torch.distributed as dist

        from yolox.data import (
            TrainTransform,
            YoloBatchSampler,
            DataLoader,
            InfiniteSampler,
            MosaicDetection,
            worker_init_reset_seed,
        )
        from yolox.utils import wait_for_the_master

        if self.dataset is None:
            with wait_for_the_master():
                assert cache_img is None, (
                    "cache_img must be None if you didn't create self.dataset before launch"
                )
                self.dataset = self.get_dataset(cache=False, cache_type=cache_img)

        self.dataset = MosaicDetection(
            dataset=self.dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=4000, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob
            ),
            degrees=self.degrees,
            translate=self.translate,
            mosaic_scale=self.mosaic_scale,
            mixup_scale=self.mixup_scale,
            shear=self.shear,
            enable_mixup=self.enable_mixup,
            mosaic_prob=self.mosaic_prob,
            mixup_prob=self.mixup_prob,
        )

        if is_distributed:
            batch_size = batch_size // dist.get_world_size()

        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            mosaic=not no_aug,
        )
        dataloader_kwargs = {
            "num_workers": self.data_num_workers,
            "pin_memory": True,
            "batch_sampler": batch_sampler,
            "worker_init_fn": worker_init_reset_seed,
        }
        return DataLoader(self.dataset, **dataloader_kwargs)

    def get_eval_dataset(self, **kwargs):
        from yolox.data import COCODataset, ValTransform

        legacy = kwargs.get("legacy", False)
        dataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            name=self.image_folder,
            img_size=self.test_size,
            preproc=ValTransform(legacy=legacy),
        )
        assert_class_scheme(dataset.coco, self.num_classes)
        return dataset

    def get_evaluator(self, batch_size, is_distributed, testdev=False, legacy=False):
        return VisDroneEvaluator(
            dataloader=self.get_eval_loader(
                batch_size, is_distributed, testdev=testdev, legacy=legacy
            ),
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )


In [ ]:
# Bagli veri setlerini bul ve tek bir 2-sinifli COCO'ya birlestir.
# build_dataset.py sinif eslemesini, oturum bazli bolmeyi ve dogrulamayi
# kendisi yapar; bozuk bir sey varsa dosya uretmeden hata verip durur.
INPUT = Path("/kaggle/input")


def find_source(*patterns, label=""):
    """Bagli veri setleri icinde ilk eslesen yolu dondurur (zip veya klasor)."""
    for pattern in patterns:
        for path in sorted(INPUT.glob(pattern)):
            return path
    raise SystemExit(
        f"'{label}' bulunamadi. Aranan desenler: {patterns}. "
        f"Add Input ile ilgili veri setini bagladiginizdan emin olun."
    )


SOURCES = {
    "visdrone-train": find_source(
        "**/VisDrone2019-DET-train/VisDrone2019-DET-train",
        "**/VisDrone2019-DET-train",
        "**/VisDrone2019-DET-train.zip",
        label="VisDrone train"),
    "visdrone-val": find_source(
        "**/VisDrone2019-DET-val/VisDrone2019-DET-val",
        "**/VisDrone2019-DET-val",
        "**/VisDrone2019-DET-val.zip",
        label="VisDrone val"),
    "vesselimg": find_source("**/VESSELimg*", label="VESSELimg"),
    "milrec": find_source("**/Military*Vehicle*Recognition*",
                          label="Military Vehicle Recognition"),
    "mendeley": find_source("**/*Multi-Class*UAV*Military*",
                            label="Mendeley UAV Military"),
}
for _name, _path in SOURCES.items():
    print(f"{_name:<16} {_path}")

DATASET_DIR = Path(WORK) / "datasets" / "merged"
_args = " ".join(f'--source {k}="{v}"' for k, v in SOURCES.items())
!python build_dataset.py {_args} --out "{DATASET_DIR}" --images-out "{DATASET_DIR}/images"


In [ ]:
# Baslangic agirligi: YOLOX'un resmi Megvii COCO checkpoint'i.
# COCO'da hem arac hem `boat` siniflari var; bu fine-tune icin dogrudan
# uygun bir baslangic noktasi.
import importlib.util
import urllib.request

WDIR = Path(WORK) / "weights"
WDIR.mkdir(exist_ok=True)

spec = importlib.util.spec_from_file_location("exp_mod", "yolox_nano_visdrone.py")
exp_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_mod)
ref_state = exp_mod.Exp().get_model().state_dict()

mg = WDIR / "yolox_nano.pth"
if not mg.exists():
    urllib.request.urlretrieve(
        "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth",
        mg,
    )

# PyTorch 2.6+ varsayilani weights_only=True, Vitis AI 3.0/PyTorch 1.12 ise
# bu parametreyi tanimaz. Iki ortamda da acik ve guvenilir yerel ckpt yukle.
try:
    raw = torch.load(mg, map_location="cpu", weights_only=False)
except TypeError:
    raw = torch.load(mg, map_location="cpu")
inner = raw.get("model", raw.get("state_dict", raw)) if isinstance(raw, dict) else raw
inner = {(k[7:] if k.startswith("module.") else k): v for k, v in inner.items()}

matched = {
    k: v for k, v in inner.items()
    if k in ref_state and ref_state[k].shape == v.shape
}
missing = sorted(set(ref_state) - set(matched))
allowed_missing = all(
    k == "backbone.backbone.stem.space_to_depth.weight" or ".cls_preds." in k
    for k in missing
)
ratio = len(matched) / max(len(ref_state), 1)
assert ratio >= 0.95 and allowed_missing, (
    f"Baslangic checkpoint'i mimariyle uyumsuz: eslesme={ratio:.1%}, "
    f"beklenmeyen eksikler={missing}"
)

# 2 sinifli head ve sabit DPUFocus agirligi yeni modelin baslangicindan,
# diger katmanlar Megvii'den gelir.
init_state = dict(ref_state)
init_state.update(matched)
INIT_CKPT = str(WDIR / "init_ckpt.pth")
torch.save({
    "model": init_state,
    "meta": {"source": str(mg), "yolox_commit": YOLOX_COMMIT,
             "matched_ratio": ratio},
}, INIT_CKPT)
print(f"Megvii baslangici dogrulandi: {ratio:.1%} -> {INIT_CKPT}")
print("Yalnizca cls head (2 sinif) ve sabit DPUFocus katmani yeniden baslatildi.")


## Egitim

In [ ]:
# 2 sinif (land_vehicle / sea_vehicle), girdi 896x512 (16:9), 40 epoch.
# 10 sinifli referans kosumda model epoch ~40'ta doymustu; 80 yerine 40 epoch.
BATCH = 16  # OOM olursa 8 yapin

%cd {WORK}
!python YOLOX/tools/train.py -f yolox_nano_visdrone.py -d 1 -b {BATCH} --fp16 -c weights/init_ckpt.pth


In [ ]:
# Devam (resume): onceki oturumun Output'unu bu oturuma input olarak
# bagladiktan sonra asagidaki degiskene latest_ckpt.pth yolunu yazin.
RESUME_CKPT = ""  # or. "/kaggle/input/ONCEKI/YOLOX_outputs/yolox_nano_visdrone/latest_ckpt.pth"

if RESUME_CKPT:
    !python YOLOX/tools/train.py -f yolox_nano_visdrone.py -d 1 -b {BATCH} --fp16 --resume -c "{RESUME_CKPT}"
else:
    print("RESUME_CKPT bos; bu hucre yalnizca yarim kalan egitimi surdurmek icin.")


In [ ]:
# Degerlendirme: resmi VisDrone ignore filtresi + global top-500 + VOC AP,
# ayrica sinif bazli AP ve P/R/F1. Bu sayilari not edin: kuantalama sonrasi
# ayni protokolle karsilastirilacak.


def find_best_ckpt():
    """Bu oturumda egitildiyse yerelden, aksi halde bagli input'tan alir."""
    local = Path("YOLOX_outputs/yolox_nano_visdrone/best_ckpt.pth")
    if local.exists():
        return local
    if INPUT.exists():
        for candidate in sorted(INPUT.glob("**/best_ckpt.pth")):
            return candidate
    raise SystemExit(
        "best_ckpt.pth bulunamadi. Once egitim hucresini calistirin veya "
        "onceki oturumun ciktisini bu oturuma input olarak baglayin."
    )


BEST_CKPT = str(find_best_ckpt())
print("checkpoint:", BEST_CKPT)
!python YOLOX/tools/eval.py -f yolox_nano_visdrone.py -c "{BEST_CKPT}" -d 1 -b 16 --conf 0.001


In [ ]:
# Gorsel kontrol: kutular + MERKEZ NOKTALARI (cx, cy)
# KV260 uygulamasindaki hesabin aynisi: cx=(x1+x2)/2, cy=(y1+y2)/2
import json
import random

import cv2
import matplotlib.pyplot as plt

from yolox.data.data_augment import ValTransform
from yolox.utils import postprocess

spec2 = importlib.util.spec_from_file_location("exp_mod2", "yolox_nano_visdrone.py")
exp_mod2 = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(exp_mod2)
exp = exp_mod2.Exp()
CLASSES = exp_mod2.TARGET_CLASSES

model = exp.get_model().eval()
try:
    ckpt = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(BEST_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

CONF_VIS = 0.15  # kartla ayni dagitim esigi
val_transform = ValTransform(legacy=False)

# Her kaynaktan birer ornek: iki sinifi da gorelim
val_json = json.loads(
    (DATASET_DIR / "annotations" / "instances_val.json").read_text())
by_source = {}
for _image in val_json["images"]:
    by_source.setdefault(_image["source"], []).append(_image["file_name"])
random.seed(0)
sample_names = [random.choice(v) for v in by_source.values()]

fig, axes = plt.subplots(len(sample_names), 1,
                         figsize=(16, 9 * len(sample_names)))
axes = np.atleast_1d(axes)
for ax, name in zip(axes, sample_names):
    img0 = cv2.imread(str(DATASET_DIR / "images" / name))
    h0, w0 = img0.shape[:2]
    ratio = min(exp.test_size[0] / h0, exp.test_size[1] / w0)
    img, _ = val_transform(img0, None, exp.test_size)
    with torch.no_grad():
        out = model(torch.from_numpy(img).unsqueeze(0).float().to(device))
        out = postprocess(out, exp.num_classes, CONF_VIS, exp.nmsthre)[0]
    vis = img0.copy()
    if out is not None:
        for *box, obj_conf, cls_conf, cls_id in out.cpu().numpy():
            x1, y1, x2, y2 = [v / ratio for v in box]
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            score = obj_conf * cls_conf
            cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)),
                          (0, 255, 0), 2)
            cv2.circle(vis, (int(cx), int(cy)), 4, (0, 0, 255), -1)
            cv2.putText(
                vis,
                f"{CLASSES[int(cls_id)]} {score:.2f} ({int(cx)},{int(cy)})",
                (int(x1), max(14, int(y1) - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Oracle VM'e tasinacak dosyalari paketle
import shutil

ART = Path(WORK) / "artifacts"
ART.mkdir(exist_ok=True)
shutil.copy(BEST_CKPT, ART / "best_ckpt.pth")
for _name in ("yolox_nano_visdrone.py", "visdrone_eval.py", "build_dataset.py"):
    shutil.copy(_name, ART / _name)
# Kuantalama ayni val setiyle olculmeli: anotasyonu da tasi
shutil.copy(DATASET_DIR / "annotations" / "instances_val.json",
            ART / "instances_val.json")
(ART / "classes.txt").write_text("\n".join(CLASSES))
(ART / "YOLOX_COMMIT.txt").write_text(YOLOX_COMMIT + "\n")
zip_path = shutil.make_archive(str(Path(WORK) / "yolox_aerial_artifacts"),
                               "zip", ART)
print("Hazir:", zip_path)
print("Not defteri kaydedilince Output sekmesinden indirebilirsiniz.")
print("VM'e ayrica val goruntuleri lazim:", DATASET_DIR / "images")


## Sonraki adim: Oracle VM'de kuantalama

`yolox_aerial_artifacts.zip` dosyasini ve `datasets/merged/images` klasorunu
VM'e tasiyin, ardindan `quantize/README.md` adimlarini izleyin.
